# Structured validation reporting

This notebook demonstrates the additive validation API exposed by `sjd`:

- `validation_report()` snapshots the object's current validation state without running validation.
- `validate_report(...)` runs the existing validation sequence once and returns a structured snapshot.
- Existing `is_ok()`, `validate(...)`, and `error_messages()` behavior remains available.

Each structured issue contains a stable `code`, a `severity`, a component `path`, and the existing human-readable `message`.

Build the module before running this notebook (copies `_core.pyd` into `python/sjd`):

`cmake --build --preset debug --target sjd_python`

Install the package once with `python -m pip install -e python`.

Shut down the notebook kernel before rebuilding so Windows can overwrite `_core.pyd`.

In [1]:
from datetime import date

import sjd
from sjd import examples

print(sjd.__file__)
print("Structured validation types:", sjd.ValidationIssue, sjd.ValidationReport)


E:\Code\sjd\python\sjd\__init__.py
Structured validation types: <class 'sjd._core.ValidationIssue'> <class 'sjd._core.ValidationReport'>


In [2]:
def show_report(title, report):
    print(f"\n{title}")
    print("OK:", report.ok)
    if report.issues:
        print(f"{'Code':<42} {'Severity':<9} {'Path':<55} Message")
        for issue in report.issues:
            path = ".".join(issue.path)
            print(
                f"{issue.code:<42} {issue.severity.name:<9} "
                f"{path:<55} {issue.message}"
            )
    else:
        print("No issues")


## Snapshot versus validation

A newly created rates environment has not yet been validated. Its snapshot is therefore empty. Calling `validate_report()` runs the existing check and reports that no rates have been supplied.

In [3]:
anchor = date(2026, 9, 16)
empty_environment = sjd.RatesEnvironment(anchor)

show_report("Snapshot before validation", empty_environment.validation_report())
empty_report = empty_environment.validate_report()
show_report("Report after validation", empty_report)

assert empty_report.issues[0].code == "RATES_ENVIRONMENT_NO_RATES"
assert empty_report.issues[0].path == ("rates_environment",)
print("Legacy API:", empty_environment.is_ok(), empty_environment.error_messages())


Snapshot before validation
OK: True
No issues

Report after validation
OK: False
Code                                       Severity  Path                                                    Message
RATES_ENVIRONMENT_NO_RATES                 ERROR     rates_environment                                       No rates set
Legacy API: False RatesEnvironment: No rates set


## Several errors from one contract

The forward has a negative strike and the rates environment is empty. The structured result distinguishes the contract input from its missing market-data dependencies.

In [4]:
invalid_forward = sjd.EuropeanForward(
    settlement=date(2026, 12, 16),
    strike=-1.0,
    volume=1_000_000.0,
    buy_sell=sjd.BuySell.BUY,
)

# A snapshot still does not trigger validation.
show_report("Forward snapshot before validation", invalid_forward.validation_report())

forward_report = invalid_forward.validate_report(empty_environment)
show_report("Invalid forward and market setup", forward_report)

assert [issue.code for issue in forward_report.issues] == [
    "CONTRACT_STRIKE_NEGATIVE",
    "RATES_ENVIRONMENT_NO_RATES",
    "MARKET_DISCOUNT_RATE_SOURCE_MISSING",
    "MARKET_FORWARD_RATE_SOURCE_MISSING",
]
assert forward_report.issues[0].path == ("european_forward", "strike")


Forward snapshot before validation
OK: True
No issues

Invalid forward and market setup
OK: False
Code                                       Severity  Path                                                    Message
CONTRACT_STRIKE_NEGATIVE                   ERROR     european_forward.strike                                 Strike is negative
RATES_ENVIRONMENT_NO_RATES                 ERROR     european_forward.rates_environment                      No rates set
MARKET_DISCOUNT_RATE_SOURCE_MISSING        ERROR     european_forward.rates_environment.discount_rate_source No discounting rate source
MARKET_FORWARD_RATE_SOURCE_MISSING         ERROR     european_forward.rates_environment.forward_rate_source  No forward rate source


## Nested market-data failure

Here the environment contains a discount curve, but the curve's date and discount-factor vectors have different lengths. The curve records its own code, and the rates environment prefixes the path.

In [5]:
invalid_discount_curve = sjd.DiscountCurve(
    anchor_date=anchor,
    observation_dates=[anchor, date(2027, 9, 16)],
    discount_factors=[1.0],
    allow_extrapolation=False,
)
invalid_market = sjd.RatesEnvironment(anchor)
invalid_market.set_drs(invalid_discount_curve)

nested_report = invalid_market.validate_report()
show_report("Invalid nested discount curve", nested_report)

assert nested_report.issues[0].code == "DISCOUNT_CURVE_DIMENSION_MISMATCH"
assert nested_report.issues[0].path == (
    "rates_environment",
    "discount_rate_source",
    "discount_curve",
    "inputs",
)


Invalid nested discount curve
OK: False
Code                                       Severity  Path                                                    Message
DISCOUNT_CURVE_DIMENSION_MISMATCH          ERROR     rates_environment.discount_rate_source.discount_curve.inputs ArrayInterpolator::ArrayInterpolator. X and Y vectors do not have the same dimension


## Corrected setup

Construct valid forward and discount curves, then validate a replacement forward with a positive strike. The report is now ready (`ok == True`) and contains no issues.

In [6]:
curve_dates = [anchor, date(2026, 12, 16), date(2027, 9, 16)]
forwards, discount_factors, _volatilities = examples.fx_curve_values(curve_dates)

frs = sjd.ForwardCurve(anchor, curve_dates, forwards, allow_extrapolation=False)
drs = sjd.DiscountCurve(
    anchor, curve_dates, discount_factors, allow_extrapolation=False
)
environment = sjd.RatesEnvironment(anchor)
environment.set_frs(frs)
environment.set_drs(drs)

valid_forward = sjd.EuropeanForward(
    settlement=date(2026, 12, 16),
    strike=17.60,
    volume=1_000_000.0,
    buy_sell=sjd.BuySell.BUY,
)
ready_report = valid_forward.validate_report(environment)
show_report("Corrected forward setup", ready_report)

assert ready_report.ok
assert ready_report.issues == ()
print("Forward value:", valid_forward.get_value(environment))



Corrected forward setup
OK: True
No issues
Forward value: 30838.214001485692
